# Evaluate AI models on text Language Identification (LLMs, langdetect, GlotLID, NLLB-LID)

This notebook runs an **extensive comparison** on CommonLID: three standard LID python tools plus several LLMs, each LLM evaluated **zero-, one-, and five-shot**. Every model is scored on the *same* sampled rows, and the few-shot demonstrations are held out from a fixed pool so they never leak into the evaluation set.

**Before running the LLMs** you will need a way to run LLM inference. You can do it either with local LLMs, or a 3rd party provider. In this notebook we are using Together AI, so we will set up a `TOGETHER_API_KEY=...` in the repo's `.env` (the setup cell loads it). Standard tools need no key. Any model that errors (missing key, unavailable model id, …) is skipped and reported, so the rest of the comparison still runs.

**This evaluation can become very computationally expensive due to the LLM usage!** For example, 100 samples/language (~8.7k rows) × (3 standard + 5 LLMs × 3 shots) = 18 runs, i.e. tens of thousands of serial API calls. Each run is saved to `results/` as it finishes, so a crash mid-way doesn't lose completed work, and you can reload everything later with `compare-saved-runs.ipynb`.

## Setup

In [1]:
# Auto-reload edited library code (e.g. language_id.models.together) without a kernel restart.
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# Load TOGETHER_API_KEY from the repo .env so the LLM models can authenticate.
try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../../.env'))
except Exception as e:
    print('dotenv not loaded:', e)

from language_id.data import load_commonlid, load_commonvoice_lid, sample, take_fewshot_examples
from language_id.evaluate import evaluate
from language_id.models import get_model, TOGETHER_MODELS
from language_id.models.together import TogetherModel
from language_id.reporting import save_run

## Choose what to compare

- `STANDARD`: Python libraries that offer text LID out-of-the-box (langdetect, GlotLID, NLLB-LID). They run locally with no API key and are always zero-shot.
- `LLMS`: Together-hosted models (need `TOGETHER_API_KEY`). Each is evaluated at every value in `SHOTS`.
- `SHOTS`: few-shot settings for the LLMs. `0` is zero-shot; `k` prepends `k` worked examples to the prompt.
- `N_PER_LANG`: rows sampled per language (the eval set).
- `MAX_OUTPUT_TOKENS`: output-token cap per LLM call. Reasoning models spend output tokens on hidden reasoning, so this bounds cost; keep it high enough to leave room for the answer.
- `MAX_LLM_WORKERS`: concurrent API requests per LLM run. Higher is faster; lower it if you hit rate limits (1 = serial).
- `SAVE`: write each run to `results/` (so it can be reloaded by `compare-saved-runs.ipynb`).

In [ ]:
DATASET = 'commonlid'  # 'commonlid' or 'commonvoice_lid'
N_PER_LANG = 200       # rows per language to evaluate on, -1 to evaluate on the whole dataset
SEED = 0

STANDARD = ['langdetect', 'glotlid', 'nllb-lid']
LLMS = ['gpt-oss-120b', "qwen"] # 'qwen', 'gpt-oss-120b', 'gemma', 'gpt-oss-20b', 'llama']
SHOTS = [0, 1]      # few-shot settings applied to every LLM (standard tools are always 0-shot)

MAX_OUTPUT_TOKENS = 256       # output-token cap per LLM call
MAX_LLM_WORKERS = 8        # concurrent LLM requests per run
SAVE = True            # persist each run to results/
RESULTS_DIR = Path('../../../results')

# Comparison results from this notebook are written here
UNIQUE_EXPERIMENT_NAME = f"{DATASET}_{N_PER_LANG}_{SEED}_{datetime.now(UTC).strftime('%Y%m%d_T%H%M%S')}"
COMPARISON_DIR = RESULTS_DIR / 'comparisons' / UNIQUE_EXPERIMENT_NAME
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

## Load the dataset, hold out few-shot examples, and sample the eval set

The few-shot demonstrations are drawn first (spanning distinct languages) and removed from the pool, then the evaluation set is sampled from what remains. This guarantees the eval rows are identical for every model and shot count, and that no demonstration is ever scored.

In [3]:
full = load_commonlid() if DATASET == 'commonlid' else load_commonvoice_lid(split='test')

# Hold out a fixed pool of demonstrations (the largest shot count we'll use),
# then sample the eval set from the rest. k-shot uses the first k of these.
fewshot_pool, remaining = take_fewshot_examples(full, max(SHOTS, default=0), seed=SEED)
df = sample(remaining, n_per_lang=N_PER_LANG, seed=SEED)

print(f'{DATASET}: {len(df)} eval rows across {df["lang"].nunique()} languages '
      f'(sampled {N_PER_LANG}/language from {len(full)} total)')
print(f'few-shot pool ({len(fewshot_pool)} examples, distinct languages): '
      f'{[code for _, code in fewshot_pool]}')

commonlid: 16119 eval rows across 108 languages (sampled 200/language from 373230 total)
few-shot pool (1 examples, distinct languages): ['vie']


## Run the evaluations

Standard tools run once (zero-shot); each LLM runs once per value in `SHOTS`. Results are labelled `model` (standard) or `model-Kshot` (LLMs) so the shot settings compare side by side. Failures (e.g. missing API key) are caught and listed so the comparison still proceeds with whatever succeeded.

In [ ]:
from tqdm.auto import tqdm

overall_rows = []
per_lang_frames = []
kinds = {}        # label -> 'standard' | 'LLM'
failures = {}

total_runs = len(STANDARD) + len(LLMS) * len(SHOTS)
n_rows = len(df)
run_idx = 0


def _record(label, kind, model_obj):
    global run_idx
    run_idx += 1
    # Per-row progress bar: predict_batch calls predict once per row, so wrapping
    # predict lets the bar tick as each row finishes (even across worker threads).
    bar = tqdm(total=n_rows, desc=f'[{run_idx}/{total_runs}] {label}', leave=False)
    base_predict = model_obj.predict

    def predict_with_progress(text):
        try:
            return base_predict(text)
        finally:
            bar.update(1)

    model_obj.predict = predict_with_progress
    t = time.time()
    try:
        overall, per_lang, predictions = evaluate(df, model_obj)
    finally:
        bar.close()
    overall['model'] = label
    per_lang['model'] = label
    overall_rows.append(overall)
    per_lang_frames.append(per_lang)
    kinds[label] = kind
    if SAVE:
        save_run(COMPARISON_DIR, label, DATASET, overall, per_lang, predictions)
    print(f"[{run_idx}/{total_runs}] {label:22} acc={overall['accuracy']:.3f}  macroF1={overall['macro_f1']:.3f}  ({time.time() - t:.0f}s)")


# 1) Standard tools: local, zero-shot.
for name in STANDARD:
    try:
        _record(name, 'standard', get_model(name))
    except Exception as e:
        failures[name] = repr(e)
        print('  FAILED:', e)

# 2) LLMs: each shot setting.
for name in LLMS:
    for shot in SHOTS:
        label = f'{name}-{shot}shot'
        try:
            model_obj = TogetherModel(
                model_id=TOGETHER_MODELS[name],
                name=label,
                examples=fewshot_pool[:shot],
                max_output_tokens=MAX_OUTPUT_TOKENS,
                max_workers=MAX_LLM_WORKERS,
            )
            _record(label, 'LLM', model_obj)
        except Exception as e:
            failures[label] = repr(e)
            print('  FAILED:', e)

if failures:
    print('\nskipped:', failures)
assert overall_rows, 'no model succeeded — check API key / model availability'


/home/kostis/Projects/MDC/language-id/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/7] langdetect             acc=0.394  macroF1=0.240  (63s)


[2/7] glotlid                acc=0.756  macroF1=0.702  (29s)


[3/7] nllb-lid               acc=0.629  macroF1=0.518  (16s)


[4/7] gpt-oss-120b-0shot     acc=0.653  macroF1=0.582  (4663s)


[5/7] gpt-oss-120b-1shot:  93%|█████████▎| 14943/16119 [49:16<04:50,  4.05it/s]  

## Overview comparison table

One row per run, sorted by macro-F1.

In [ ]:
overall_df = (
    pd.DataFrame(overall_rows)[['model', 'accuracy', 'macro_f1', 'n', 'n_languages']]
    .sort_values('macro_f1', ascending=False)
    .reset_index(drop=True)
)
overall_df.insert(1, 'kind', overall_df['model'].map(kinds))
overall_df.style.format({'accuracy': '{:.3f}', 'macro_f1': '{:.3f}'}).background_gradient(
    subset=['accuracy', 'macro_f1'], cmap='Greens', vmin=0, vmax=1
)
overall_df.to_csv(COMPARISON_DIR / f'overall_comparison_{DATASET}.csv', index=False)

## Overview comparison figure

In [ ]:
plot_df = overall_df.set_index('model')[['accuracy', 'macro_f1']]
ax = plot_df.plot.bar(figsize=(max(8, 0.7 * len(plot_df)), 4.5), rot=60, ylim=(0, 1))
for tick, label in zip(ax.get_xticklabels(), plot_df.index):
    tick.set_ha('right')
    tick.set_color('#C44E52' if kinds.get(label) == 'LLM' else '#4C72B0')
ax.set_ylabel('score')
ax.set_title(f'Standard tools vs. LLMs (few-shot) — {DATASET} ({N_PER_LANG}/lang)')
ax.legend(title='metric')
plt.tight_layout()
fig_path = COMPARISON_DIR / f'overview_{DATASET}.png'
ax.figure.savefig(fig_path, dpi=150, bbox_inches='tight')
print('saved', fig_path)
plt.show()

## Per-language comparison

Per-language metrics for every run. `support` is the number of gold samples used per language.

In [ ]:
per_lang_all = pd.concat(per_lang_frames, ignore_index=True)
METRIC = 'f1'  # try 'accuracy' or 'precision'
pivot = per_lang_all.pivot_table(index=['name', 'lang'], columns='model', values=METRIC)
support = per_lang_all.groupby(['name', 'lang'])['support'].max()
pivot.insert(0, 'support', support)
pivot = pivot.sort_values('support', ascending=False)
model_cols = [c for c in pivot.columns if c != 'support']
pivot.style.format({**{m: '{:.2f}' for m in model_cols}, 'support': '{:.0f}'}).background_gradient(
    subset=model_cols, cmap='RdYlGn', vmin=0, vmax=1
)
per_lang_all.to_csv(COMPARISON_DIR / f'per_language_{DATASET}.csv', index=False)

### Per-language heatmap

In [ ]:
hm = pivot[model_cols]
fig, ax = plt.subplots(figsize=(max(4, 0.9 * len(model_cols)), max(4, 0.28 * len(hm))))
im = ax.imshow(hm.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(model_cols)))
ax.set_xticklabels(model_cols, rotation=45, ha='right')
ax.set_yticks(range(len(hm)))
ax.set_yticklabels([f'{n} ({l})' for n, l in hm.index], fontsize=7)
ax.set_title(f'Per-language {METRIC} — {DATASET}')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=METRIC)
plt.tight_layout()
fig_path = COMPARISON_DIR / f'per_language_{METRIC}_heatmap_{DATASET}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print('saved', fig_path)
plt.show()

### Where models disagree most

Languages with the largest best-minus-worst spread across runs.

In [ ]:
top_n = 5

if len(model_cols) >= 2:
    spread = (hm.max(axis=1) - hm.min(axis=1)).sort_values(ascending=False)
    top = spread.head(top_n).index[::-1]
    top_df = hm.loc[top]
    ax = top_df.plot.barh(figsize=(10, max(4, 0.5 * len(top_df))), xlim=(0, 1))
    ax.set_yticklabels([f'{n} ({l})' for n, l in top_df.index])
    ax.set_xlabel(METRIC)
    ax.set_title(f'Top languages by cross-model {METRIC} spread — {DATASET}')
    ax.legend(title='run', loc='lower right', fontsize=7)
    plt.tight_layout()
    fig_path = COMPARISON_DIR / f'disagreement_{METRIC}_{DATASET}.png'
    ax.figure.savefig(fig_path, dpi=150, bbox_inches='tight')
    print('saved', fig_path)
    plt.show()
else:
    print('Need at least two successful runs to compare.')